<a href="https://colab.research.google.com/github/Bhavesh43-OG/bhaveshd/blob/main/Bhavesh_Desale_Exp9_Chi_Squared_Tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

titanic_path = kagglehub.competition_download('titanic')

print('Data source import complete.')


# Python for Data 25: Chi-Squared Tests
[back to index](https://www.kaggle.com/hamelg/python-for-data-analysis-index)

Last lesson we introduced the framework of statistical hypothesis testing and the t-test for investigating differences between numeric variables. In this lesson, we turn our attention to a common statistical test for categorical variables: the chi-squared test.

# Chi-Squared Goodness-Of-Fit Test

In our study of t-tests, we introduced the one-way t-test to check whether a sample mean differs from the an expected (population) mean. The chi-squared goodness-of-fit test is an analog of the one-way t-test for categorical variables: it tests whether the distribution of sample categorical data matches an expected distribution. For example, you could use a chi-squared goodness-of-fit test to check whether the race demographics of members at your church or school match that of the entire U.S. population or whether the computer browser preferences of your friends match those of Internet uses as a whole.

When working with categorical data, the values themselves aren't of much use for statistical testing because categories like "male", "female," and "other" have no mathematical meaning. Tests dealing with categorical variables are based on variable counts instead of the actual value of the variables themselves.

Let's generate some fake demographic data for U.S. and Minnesota and walk through the chi-square goodness of fit test to check whether they are different:

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

In [ ]:
national = pd.DataFrame(["white"]*100000 + ["hispanic"]*60000 +\
                        ["black"]*50000 + ["asian"]*15000 + ["other"]*35000)


minnesota = pd.DataFrame(["white"]*600 + ["hispanic"]*300 + \
                         ["black"]*250 +["asian"]*75 + ["other"]*150)

national_table = pd.crosstab(index=national[0], columns="count")
minnesota_table = pd.crosstab(index=minnesota[0], columns="count")

print( "National")
print(national_table)
print(" ")
print( "Minnesota")
print(minnesota_table)

National
col_0      count
0               
asian      15000
black      50000
hispanic   60000
other      35000
white     100000
 
Minnesota
col_0     count
0              
asian        75
black       250
hispanic    300
other       150
white       600


Chi-squared tests are based on the so-called chi-squared statistic. You calculate the chi-squared statistic with the following formula:

$$ sum(\frac{(observed-expected)^2}{expected}) $$

In the formula, observed is the actual observed count for each category and expected is the expected count based on the distribution of the population for the corresponding category. Let's calculate the chi-squared statistic for our data to illustrate:

In [ ]:
observed = minnesota_table

national_ratios = national_table/len(national)  # Get population ratios

expected = national_ratios * len(minnesota)   # Get expected counts

chi_squared_stat = (((observed-expected)**2)/expected).sum()

print(chi_squared_stat)

col_0
count    18.194805
dtype: float64


*Note: The chi-squared test assumes none of the expected counts are less than 5.*

Similar to the t-test where we compared the t-test statistic to a critical value based on the t-distribution to determine whether the result is significant, in the chi-square test we compare the chi-square test statistic to a critical value based on the [chi-square distribution](https://en.wikipedia.org/wiki/Chi-squared_distribution). The scipy library shorthand for the chi-square distribution is chi2. Let's use this knowledge to find the critical value for 95% confidence level and check the p-value of our result:

In [ ]:
crit = stats.chi2.ppf(q = 0.95, # Find the critical value for 95% confidence*
                      df = 4)   # Df = number of variable categories - 1

print("Critical value")
print(crit)

p_value = 1 - stats.chi2.cdf(x=chi_squared_stat,  # Find the p-value
                             df=4)
print("P value")
print(p_value)

Critical value
9.487729036781154
P value
[0.00113047]


Since our chi-squared statistic exceeds the critical value, we'd reject the null hypothesis that the two distributions are the same.

You can carry out a chi-squared goodness-of-fit test automatically using the scipy function scipy.stats.chisquare():

In [ ]:
stats.chisquare(f_obs= observed,   # Array of observed counts
                f_exp= expected)   # Array of expected counts

Power_divergenceResult(statistic=array([18.19480519]), pvalue=array([0.00113047]))

The test results agree with the values we calculated above.

# Chi-Squared Test of Independence

[Independence](https://en.wikipedia.org/wiki/Independence_(probability_theory)) is a key concept in probability that describes a situation where knowing the value of one variable tells you nothing about the value of another. For instance, the month you were born probably doesn't tell you anything about which web browser you use, so we'd expect birth month and browser preference to be independent. On the other hand, your month of birth might be related to whether you excelled at sports in school, so month of birth and sports performance might not be independent.

The chi-squared test of independence tests whether two categorical variables are independent. The test of independence is commonly used to determine whether variables like education, political views and other preferences vary based on demographic factors like gender, race and religion. Let's generate some fake voter polling data and perform a test of independence:

In [ ]:
np.random.seed(10)

# Sample data randomly at fixed probabilities
voter_race = np.random.choice(a= ["asian","black","hispanic","other","white"],
                              p = [0.05, 0.15 ,0.25, 0.05, 0.5],
                              size=1000)

# Sample data randomly at fixed probabilities
voter_party = np.random.choice(a= ["democrat","independent","republican"],
                              p = [0.4, 0.2, 0.4],
                              size=1000)

voters = pd.DataFrame({"race":voter_race,
                       "party":voter_party})

voter_tab = pd.crosstab(voters.race, voters.party, margins = True)

voter_tab.columns = ["democrat","independent","republican","row_totals"]

voter_tab.index = ["asian","black","hispanic","other","white","col_totals"]

observed = voter_tab.iloc[0:5,0:3]   # Get table without totals for later use
voter_tab

,democrat,independent,republican,row_totals
asian,21,7,32,60
black,65,25,64,154
hispanic,107,50,94,251
other,15,8,15,38
white,189,96,212,497
col_totals,397,186,417,1000


Note that we did not use the race data to inform our generation of the party data so the variables are independent.

For a test of independence, we use the same chi-squared formula that we used for the goodness-of-fit test. The main difference is we have to calculate the expected counts of each cell in a 2-dimensional table instead of a 1-dimensional table. To get the expected count for a cell, multiply the row total for that cell by the column total for that cell and then divide by the total number of observations. We can quickly get the expected counts for all cells in the table by taking the row totals and column totals of the table, performing an outer product on them with the np.outer() function and dividing by the number of observations:

In [ ]:
expected =  np.outer(voter_tab["row_totals"][0:5],
                     voter_tab.loc["col_totals"][0:3]) / 1000

expected = pd.DataFrame(expected)

expected.columns = ["democrat","independent","republican"]
expected.index = ["asian","black","hispanic","other","white"]

expected

,democrat,independent,republican
asian,23.820,11.160,25.020
black,61.138,28.644,64.218
hispanic,99.647,46.686,104.667
other,15.086,7.068,15.846
white,197.309,92.442,207.249


Now we can follow the same steps we took before to calculate the chi-square statistic, the critical value and the p-value:

In [ ]:
chi_squared_stat = (((observed-expected)**2)/expected).sum().sum()

print(chi_squared_stat)

7.169321280162059


*Note: We call .sum() twice: once to get the column sums and a second time to add the column sums together, returning the sum of the entire 2D table.*

In [ ]:
crit = stats.chi2.ppf(q = 0.95, # Find the critical value for 95% confidence*
                      df = 8)   # *

print("Critical value")
print(crit)

p_value = 1 - stats.chi2.cdf(x=chi_squared_stat,  # Find the p-value
                             df=8)
print("P value")
print(p_value)

Critical value
15.50731305586545
P value
0.518479392948842


*Note: The degrees of freedom for a test of independence equals the product of the number of categories in each variable minus 1. In this case we have a 5x3 table so df = 4x2 = 8.*

As with the goodness-of-fit test, we can use scipy to conduct a test of independence quickly. Use stats.chi2_contingency() function to conduct a test of independence automatically given a frequency table of observed counts:

In [ ]:
stats.chi2_contingency(observed= observed)

(7.169321280162059, 0.518479392948842, 8, array([[ 23.82 ,  11.16 ,  25.02 ],
        [ 61.138,  28.644,  64.218],
        [ 99.647,  46.686, 104.667],
        [ 15.086,   7.068,  15.846],
        [197.309,  92.442, 207.249]]))

The output shows the chi-square statistic, the p-value and the degrees of freedom followed by the expected counts.

As expected, given the high p-value, the test result does not detect a significant relationship between the variables.

**Question and Answer**

Q1. When do we go for Chi square test ?  

Ans. We use the **Chi-square test** when we want to analyze **categorical data** and check relationships or differences between variables.


When do we go for Chi-square Test?

 1. Testing Relationship (Independence Test)

 When you want to check whether **two categorical variables are related or independent**

**Example:**

* Gender vs Product Choice
* Education Level vs Job Type

 If variables are related → dependent
 If not → independent


2. Goodness of Fit Test

When you want to check if **observed data matches expected data**

**Example:**

* Dice fairness (expected equal probability)
* Survey results vs expected distribution

3. Data Type Requirement

 Use Chi-square test when:

* Data is **categorical (nominal/ordinal)**
* Data is in **frequency/count form** (not continuous values)

4. Large Sample Size

 Prefer when:

* Sample size is **large (n > 30 typically)**
* Expected frequency in each cell should be **≥ 5**



5. Non-Parametric Test

 Used when:

* Data does **not follow normal distribution**
* No assumptions about population parameters


When NOT to Use

* For **numerical/continuous data** (use t-test, ANOVA instead)
* When expected frequencies are very small



Q2. Do we use Chi square test for independence ? Explain How ?

Ans. Yes, we **use the Chi-square test for testing independence** between two categorical variables.



How Chi-square Test for Independence Works

1. Purpose

To check whether **two categorical variables are independent or related**

Example:

* Gender vs Product Preference
* Education vs Employment Status



2. Form Hypotheses

* **H₀ (Null Hypothesis):** Variables are **independent**
* **H₁ (Alternative Hypothesis):** Variables are **dependent (associated)**

3. Create Contingency Table

Arrange observed frequencies:

|             | Category B1  | Category B2  | Total     |
| ----------- | ------------ | ------------ | --------- |
| Category A1 | O₁₁          | O₁₂          | Row Total |
| Category A2 | O₂₁          | O₂₂          | Row Total |
| Total       | Column Total | Column Total | N         |



4. Calculate Expected Frequencies

Formula:

E_{ij} = \frac{(\text{Row Total}_i)(\text{Column Total}_j)}{N}



5. Compute Chi-square Statistic

Formula:

\chi^2 = \sum \frac{(O - E)^2}{E}

Where:

* O = Observed frequency
* E = Expected frequency



6. Degrees of Freedom

Formula:
[
df = (r - 1)(c - 1)
]


7. Decision Rule

* Compare calculated χ² with table value **OR**
* Use **p-value**

If:

* χ² calculated > χ² table
  **OR**
* p-value < α (0.05)

Reject H₀ → Variables are **dependent**

Otherwise:
Accept H₀ → Variables are **independent**



Q3. Explain role of Critical value and p-value in Chi square test.

Ans. In the **Chi-square test**, both **critical value** and **p-value** are used to decide whether to **accept or reject the null hypothesis (H₀)**.


1. Role of Critical Value

 What is Critical Value?

* It is a **threshold value** taken from the **Chi-square distribution table**
* Depends on:

  * **Level of significance (α)** (e.g., 0.05)
  * **Degrees of freedom (df)**


 How it is used?

Compare:

* **Calculated χ² value** vs **Critical value**

Decision Rule:

* If χ² (calculated) **> Critical value**
  ➝ Reject H₀
* If χ² (calculated) **≤ Critical value**
  ➝ Accept H₀


Interpretation

* Helps define a **rejection region**
* Shows whether the observed difference is **significant or due to chance**



2. Role of p-value

 What is p-value?

* It is the **probability of getting the observed result (or more extreme)** assuming H₀ is true


How it is used?

Compare:

* **p-value** vs **significance level (α)**

Decision Rule:

* If **p-value < α (0.05)**
  ➝ Reject H₀
* If **p-value ≥ α**
  ➝ Accept H₀


Interpretation

* Small p-value → Strong evidence **against H₀**
* Large p-value → Weak evidence against H₀

---

Relationship Between Critical Value & p-value

* Both give **same conclusion**, just different approaches:

  * **Critical value → table-based method**
  * **p-value → probability-based method (modern approach)**



Q4. Can we use chi square test to show existence of correlation between 2 features in a dataset ?  How will you interpret test result to indicate correlation presence.

Ans. Yes — the **Chi-square test** can be used to check the **existence of association (often loosely called “correlation”)** between **two categorical features**.

Important: It does **not measure strength or direction** like Pearson correlation. It only tells whether a **relationship exists or not**.




When can we use it?

When both features are:

* **Categorical (nominal/ordinal)**
* Represented as **frequency counts**

**Example:**

* Gender vs Product Choice
* Education vs Income Group


How it indicates correlation (association)

Step 1: Set Hypotheses

* **H₀:** No association (features are independent)
* **H₁:** Association exists (features are dependent)


Step 2: Perform Chi-square Test

* Create contingency table
* Calculate χ² statistic and p-value


Step 3: Interpret Result

 Using p-value

* If **p-value < α (0.05)**
  ➝ Reject H₀
  ➝**Association exists (correlation present)**

* If **p-value ≥ α**
  ➝ Accept H₀
  ➝ **No association (no correlation)**



Using Critical Value

* If **χ² calculated > χ² critical**
  ➝ Reject H₀ → Association exists

* Else
  ➝ No association



Final Interpretation

👉 If H₀ is rejected:

* There is a **statistically significant relationship**
* We say **features are correlated (associated)**

👉 If H₀ is accepted:

* Features are **independent**
* No correlation exists




Q5. What is degrees of freedom and how to calculate it while performing chi square test.

Ans. **Degrees of Freedom (df)** is an important concept in the **Chi-square test** that represents the **number of independent values that can vary** in your data while calculating a statistic.

What is Degrees of Freedom?

It is the number of values that are **free to vary** after certain constraints (like row and column totals) are applied.

In simple terms:
It tells how much **independent information** is available to estimate variability.



How to Calculate Degrees of Freedom in Chi-square Test

For Chi-square Test of Independence

#Formula:

[
df = (r - 1)(c - 1)
]

Where:

* **r** = number of rows
* **c** = number of columns


Example

Suppose a contingency table:

|    | B1 | B2 | B3 |
| -- | -- | -- | -- |
| A1 |    |    |    |
| A2 |    |    |    |

* Rows (r) = 2
* Columns (c) = 3

👉 Degrees of freedom:
[
df = (2 - 1)(3 - 1) = 1 \times 2 = 2
]



For Goodness of Fit Test

Formula:

[
df = n - 1
]

Where:

* **n** = number of categories

